In [12]:
%%capture
!pip install pydantic-ai google-genai tenacity

In [13]:
import sqlite3
import json
import os
import re
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.google import GoogleModel
from google.colab import userdata

# 1. Configuração de Ambiente
os.environ['GEMINI_API_KEY'] = userdata.get('gemini-drm-pa')

# 2. Output Estruturado
class AnaliseResultado(BaseModel):
    sql_utilizado: str = Field(description="A query SQL executada. Se houve erro, explique.")
    conclusao: str = Field(description="A resposta em linguagem natural para o usuário.")

# 3. Definição do Agente
model = GoogleModel('gemini-2.5-flash')
agent = Agent(
    model,
    output_type=AnaliseResultado,
    system_prompt=(
        # Técnicas de engenharia de prompt utilizadas:
        # Persona prompting, context setting (tabelas) utilizando caixa alta como mostrado em aula, chain-of-though implícito e constraint setting
        "Você é um analista de dados especialista em SQL de um E-Commerce. "
        "Seu objetivo é responder perguntas de negócios consultando um banco SQLite3.\n\n"
        "TABELAS:\n"
        "dim_consumidores, dim_produtos, dim_vendedores, fat_pedidos, "
        "fat_pedido_total, fat_itens_pedidos, fat_avaliacoes_pedidos.\n\n"
        "REGRAS OBRIGATÓRIAS:\n"
        "1. Nunca adivinhe colunas. Use 'ver_schema' se tiver dúvidas.\n"
        "2. Se o usuário filtrar por categorias, status ou estados (strings), use "
        "SEMPRE 'buscar_valores_distintos' primeiro para descobrir a grafia exata salva no banco "
        "(ex: 'SP' ou 'São Paulo'?).\n"
        "3. Ao executar a query final com 'executar_sql', limite o resultado (LIMIT 50)."
    )
)

# 4. Ferramenta: Schema Linking
@agent.tool
# Para prevenção de alucinações, encaminhar para a tabela específica
async def ver_schema(ctx: RunContext, nome_tabela: str) -> str:
    """Descobre quais colunas existem dentro de uma tabela."""
    try:
        conn = sqlite3.connect("/content/banco.db")
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({nome_tabela});")
        colunas = cursor.fetchall()
        conn.close()
        return json.dumps(colunas)
    except Exception as e:
        return f"Erro: {str(e)}"

# 5. Ferramenta: Ambiguidade / Information Retrieval (Planejamento)
@agent.tool
# Seguindo a lógica que Yann explicou, protege cláusulas WHERE de ambiguidades
async def buscar_valores_distintos(ctx: RunContext, nome_tabela: str, nome_coluna: str) -> str:
    """Traz exemplos reais de dados de uma coluna para evitar erros de ambiguidade."""
    try:
        conn = sqlite3.connect("/content/banco.db")
        cursor = conn.cursor()
        # Guardrail para previnir injeção básica, limpando os nomes pra deixar apenas números, letras e "_"
        tabela = re.sub(r'[^a-zA-Z0-9_]', '', nome_tabela)
        coluna = re.sub(r'[^a-zA-Z0-9_]', '', nome_coluna)

        cursor.execute(f"SELECT DISTINCT {coluna} FROM {tabela} LIMIT 30;") # Suficiente para entender o padrão sem estourar tokens
        valores = [row[0] for row in cursor.fetchall() if row[0] is not None]
        conn.close()
        return json.dumps(valores) # Retorna para a IA entender, como em estados utilando siglas
    except Exception as e:
        return f"Erro: {str(e)}"

# 6. Ferramenta: Guardrails em execução
@agent.tool
async def executar_sql(ctx: RunContext, query: str) -> str:
    """Executa a query SQL final no banco de dados."""

    # GUARDRAIL ANTI-INJECTION
    query_upper = query.upper()
    palavras_proibidas = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE', 'GRANT', 'REVOKE']

    if any(palavra in query_upper for palavra in palavras_proibidas):
        return "Erro de Segurança: A query contém comandos destrutivos não permitidos, o agente é voltado apenas para análise e não manipulação."

    try:
        conn = sqlite3.connect("/content/banco.db")
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute(query)
        resultados = [dict(row) for row in cursor.fetchall()]
        conn.close()
        return json.dumps(resultados)
    except Exception as e:
        return f"Erro na query SQL: {str(e)}. Corrija a sintaxe SQLite3 e tente novamente."

# ---------- Execução Principal (Interativa) ----------
import asyncio
from tenacity import retry, wait_exponential, stop_after_attempt

# Função com Backoff Exponencial
@retry(
    wait=wait_exponential(multiplier=2, min=3, max=15), # Espera 3s, depois 6s, 12s, até no máx 15s
    stop=stop_after_attempt(5), # Tenta no máximo 5 vezes
    reraise=True # Se falhar 5 vezes, mostra o erro real
)

async def executar_com_retry(pergunta: str):
    """Tenta rodar o agente. Se a API do Google der limite (429), ela espera e tenta de novo sozinha."""
    return await agent.run(pergunta)

async def chat_interativo():
    print("Rocket Lab 26.1 - Agente de Dados (Digite 'sair' para encerrar)")
    print("---------------------------------------------------------------")

    while True:
        pergunta = input("\nSua pergunta: ")

        if pergunta.strip().lower() in ['sair', 'exit', 'quit']:
            print("Encerrando o agente...")
            break

        if not pergunta.strip():
            continue

        try:
            print("Analisando o banco de dados...")
            resultado = await executar_com_retry(pergunta)

            print(f"\nSQL GERADO:")
            print(resultado.output.sql_utilizado)

            print(f"\nANÁLISE:")
            print(resultado.output.conclusao)

        except Exception as e:
            print(f"\nOcorreu um erro na requisição: {str(e)}")
            print("Tente novamente ou verifique os limites da API.")
            print("Em caso de erro 503 apenas aguarde um pouco e reenvie.")

        await asyncio.sleep(4)

# Para rodar a função assíncrona no Colab:
await chat_interativo()

Rocket Lab 26.1 - Agente de Dados (Digite 'sair' para encerrar)
---------------------------------------------------------------

Sua pergunta: sair
Encerrando o agente...
